# Evaluation

In [19]:
# mount google drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## File preparation to simulate final situation

Es wird eine Inputdatei für jede Task zum Testen generiert, sowie eine Golddatei pro Task gegen die die Ergebnisse verglichen werden können.

In [20]:
def clean_comments(data,column):
    data[column] = data[column].fillna(' ') # NaN Leerzeichen String ersetzen
    data[column] = data[column].apply(lambda x: ' ' if str(x).strip() == '' else x)
    return data


In [21]:
# adds the column "id" with  "document" + "_" + "comment_id"
def make_id_column(dataframe):
    dataframe["id"] = dataframe.apply(lambda x: x["document"] + "_" + str(x["comment_id"]), axis=1)
    return dataframe

In [22]:
import pandas as pd
import os
import csv
from urllib.request import urlopen

mode = "train"# "test" #

google_path = "Input/Data/"
#auskommentieren wenn lokal
google_path = "/content/drive/MyDrive/FlauschData/"

path = google_path + mode +"/"
submission_path = google_path + "submission/"


if mode == "train":
    comment = pd.read_csv(path + "comments_extended.csv").reset_index(drop=True)
    comment = clean_comments(comment,"comment")
    comment = clean_comments(comment,"translated")
    comment = clean_comments(comment,"spelling_corrected")
    comment = make_id_column(comment)

    data1 = pd.read_json(path + "test_task1.json").reset_index(drop=True)
    data2 = pd.read_json(path + "test_task2.json").reset_index(drop=True)
    task1 = pd.read_csv(path + "task1.csv").reset_index(drop=True)
    task2 = pd.read_csv(path + "task2.csv").reset_index(drop=True)
    test_indices_task1 = [(x,y) for x,y in zip(data1["document"],data1["comment_id"])] # indices of our test set data
    test_data_task1 = comment[comment.set_index(["document", "comment_id"]).index.isin(test_indices_task1)].copy().reset_index(drop=True)
    gold_data_task1 = task1[task1.set_index(["document", "comment_id"]).index.isin(test_indices_task1)].copy().reset_index(drop=True)
    gold_data_task1.to_csv(path + "gold_data_task1.csv", index=False, quoting=csv.QUOTE_ALL)

    test_indices_task2 = [(x,y) for x,y in zip(data2["document"],data2["comment_id"])] # indices of our test set data
    test_data_task2 = comment[comment.set_index(["document", "comment_id"]).index.isin(test_indices_task2)].copy().reset_index(drop=True)
    gold_data_task2 = task2[task2.set_index(["document", "comment_id"]).index.isin(test_indices_task2)].copy().reset_index(drop=True)
    gold_data_task2.to_csv(path + "gold_data_task2.csv", index=False, quoting=csv.QUOTE_ALL)
    test_data_task2 = test_data_task1.copy().reset_index(drop=True) # TODO PRÜFE ob das ok ist

if mode == "test":
    comment = pd.read_csv(path + "comments_extended.csv").reset_index(drop=True)
    comment = clean_comments(comment,"comment")
    comment = clean_comments(comment,"translated")
    comment = clean_comments(comment,"spelling_corrected")
    comment = make_id_column(comment)
    test_data_task1 = comment.reset_index(drop=True)
    test_data_task2 = comment.reset_index(drop=True)



In [23]:
test_data_task1["spelling_corrected"].isna().sum(), test_data_task1["translated"].isna().sum(), test_data_task1["comment"].isna().sum()

(np.int64(0), np.int64(0), np.int64(0))

In [24]:
test_data_task1.shape, test_data_task2.shape, comment.shape

((3706, 17), (3706, 17), (37057, 17))

## Official evaluation script

In [25]:
!pip install multiset

In [26]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from multiset import *

# Labels for the fine grained classification
ALL_LABELS = ["affection declaration","agreement","ambiguous",
              "compliment","encouragement","gratitude","group membership",
              "implicit","positive feedback","sympathy"]

# TASK 1
def binary_eval(file_gold, file_pred):

    test_g = pd.read_csv(file_gold)
    test_p = pd.read_csv(file_pred)

    (bin_prec, bin_rec, bin_f, bin_supp) = precision_recall_fscore_support(test_g.flausch, test_p.flausch, pos_label="yes", average='binary')

    print("TASK 1: BINARY CLASSIFICATION",
          "\n=============================",
          "\nPrecision:\t %.4f" % bin_prec,
          "\nRecall:\t\t %.4f" % bin_rec,
          "\nF-score:\t %.4f" % bin_f)

    return((bin_prec, bin_rec, bin_f))


def fine_grained_flausch_by_label(file_gold, file_pred):

    # read files
    gold = pd.read_csv(file_gold)
    gold['cid']= gold['document']+"_"+gold['comment_id'].apply(str)
    predicted = pd.read_csv(file_pred)
    predicted['cid']= predicted['document']+"_"+predicted['comment_id'].apply(str)


    # annotation sets (predicted)
    pred_spans = Multiset()
    pred_spans_loose = Multiset()
    pred_types = Multiset()

    # annotation sets (gold)
    gold_spans = Multiset()
    gold_spans_loose = Multiset()
    gold_types = Multiset()

    for row in predicted.itertuples(index=False):
        pred_spans.add((row.cid,row.type,row.start,row.end))
        pred_spans_loose.add((row.cid,row.start,row.end))
        pred_types.add((row.cid,row.type))
    for row in gold.itertuples(index=False):
        gold_spans.add((row.cid,row.type,row.start,row.end))
        gold_spans_loose.add((row.cid,row.start,row.end))
        gold_types.add((row.cid,row.type))

    # precision = true_pos / true_pos + false_pos
    # recall = true_pos / true_pos + false_neg
    # f_1 = 2 * prec * rec / (prec + rec)

    results = {'TOTAL': {'STRICT': {},'SPANS': {},'TYPES': {}}}
    # label-wise evaluation (only for strict and type)
    for label in ALL_LABELS:
        results[label] = {'STRICT': {},'TYPES': {}}
        gold_spans_x = set(filter(lambda x: x[1].__eq__(label), gold_spans))
        pred_spans_x = set(filter(lambda x: x[1].__eq__(label), pred_spans))
        gold_types_x = set(filter(lambda x: x[1].__eq__(label), gold_types))
        pred_types_x = set(filter(lambda x: x[1].__eq__(label), pred_types))

        # strict: spans + type must match
        ### NOTE: x and y / x returns 0 if x = 0 and y/x otherwise (test for zero division)
        strict_p = float(len(pred_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(pred_spans_x)
        strict_r = float(len(gold_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(gold_spans_x)
        strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
        results[label]['STRICT']['prec'] = strict_p
        results[label]['STRICT']['rec'] = strict_r
        results[label]['STRICT']['f1'] = strict_f

        # detection mode: only types must match (per post)
        types_p = float(len(pred_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(pred_types_x)
        types_r = float(len(gold_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(gold_types_x)
        types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
        results[label]['TYPES']['prec'] = types_p
        results[label]['TYPES']['rec'] = types_r
        results[label]['TYPES']['f1'] = types_f

    # Overall evaluation
    # strict: spans + type must match
    strict_p = float(len(pred_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(pred_spans)
    strict_r = float(len(gold_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(gold_spans)
    strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
    results['TOTAL']['STRICT']['prec'] = strict_p
    results['TOTAL']['STRICT']['rec'] = strict_r
    results['TOTAL']['STRICT']['f1'] = strict_f

    # spans: spans must match
    spans_p = float(len(pred_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(pred_spans_loose)
    spans_r = float(len(gold_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(gold_spans_loose)
    spans_f = (spans_p + spans_r) and 2 * spans_p * spans_r / (spans_p + spans_r)
    results['TOTAL']['SPANS']['prec'] = spans_p
    results['TOTAL']['SPANS']['rec'] = spans_r
    results['TOTAL']['SPANS']['f1'] = spans_f

    # detection mode: only types must match (per post)
    types_p = float(len(pred_types)) and float( len(gold_types.intersection(pred_types))) / len(pred_types)
    types_r = float(len(gold_types)) and float( len(gold_types.intersection(pred_types))) / len(gold_types)
    types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
    results['TOTAL']['TYPES']['prec'] = types_p
    results['TOTAL']['TYPES']['rec'] = types_r
    results['TOTAL']['TYPES']['f1'] = types_f

    print("STRICT:\n ",strict_p,strict_r,strict_f)
    print("SPANS:\n ",spans_p,spans_r,spans_f)
    print("TYPES:\n ",types_p,types_r,types_f)
    return(results)




def on_data_fine_grained_flausch_by_label(gold, predicted):

    gold['cid']= gold['document']+"_"+gold['comment_id'].apply(str)
    predicted['cid']= predicted['document']+"_"+predicted['comment_id'].apply(str)


    # annotation sets (predicted)
    pred_spans = Multiset()
    pred_spans_loose = Multiset()
    pred_types = Multiset()

    # annotation sets (gold)
    gold_spans = Multiset()
    gold_spans_loose = Multiset()
    gold_types = Multiset()

    for row in predicted.itertuples(index=False):
        pred_spans.add((row.cid,row.type,row.start,row.end))
        pred_spans_loose.add((row.cid,row.start,row.end))
        pred_types.add((row.cid,row.type))
    for row in gold.itertuples(index=False):
        gold_spans.add((row.cid,row.type,row.start,row.end))
        gold_spans_loose.add((row.cid,row.start,row.end))
        gold_types.add((row.cid,row.type))

    # precision = true_pos / true_pos + false_pos
    # recall = true_pos / true_pos + false_neg
    # f_1 = 2 * prec * rec / (prec + rec)

    results = {'TOTAL': {'STRICT': {},'SPANS': {},'TYPES': {}}}
    # label-wise evaluation (only for strict and type)
    for label in ALL_LABELS:
        results[label] = {'STRICT': {},'TYPES': {}}
        gold_spans_x = set(filter(lambda x: x[1].__eq__(label), gold_spans))
        pred_spans_x = set(filter(lambda x: x[1].__eq__(label), pred_spans))
        gold_types_x = set(filter(lambda x: x[1].__eq__(label), gold_types))
        pred_types_x = set(filter(lambda x: x[1].__eq__(label), pred_types))

        # strict: spans + type must match
        ### NOTE: x and y / x returns 0 if x = 0 and y/x otherwise (test for zero division)
        strict_p = float(len(pred_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(pred_spans_x)
        strict_r = float(len(gold_spans_x)) and float( len(gold_spans_x.intersection(pred_spans_x))) / len(gold_spans_x)
        strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
        results[label]['STRICT']['prec'] = strict_p
        results[label]['STRICT']['rec'] = strict_r
        results[label]['STRICT']['f1'] = strict_f

        # detection mode: only types must match (per post)
        types_p = float(len(pred_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(pred_types_x)
        types_r = float(len(gold_types_x)) and float( len(gold_types_x.intersection(pred_types_x))) / len(gold_types_x)
        types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
        results[label]['TYPES']['prec'] = types_p
        results[label]['TYPES']['rec'] = types_r
        results[label]['TYPES']['f1'] = types_f

    # Overall evaluation
    # strict: spans + type must match
    strict_p = float(len(pred_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(pred_spans)
    strict_r = float(len(gold_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(gold_spans)
    strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
    results['TOTAL']['STRICT']['prec'] = strict_p
    results['TOTAL']['STRICT']['rec'] = strict_r
    results['TOTAL']['STRICT']['f1'] = strict_f

    # spans: spans must match
    spans_p = float(len(pred_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(pred_spans_loose)
    spans_r = float(len(gold_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(gold_spans_loose)
    spans_f = (spans_p + spans_r) and 2 * spans_p * spans_r / (spans_p + spans_r)
    results['TOTAL']['SPANS']['prec'] = spans_p
    results['TOTAL']['SPANS']['rec'] = spans_r
    results['TOTAL']['SPANS']['f1'] = spans_f

    # detection mode: only types must match (per post)
    types_p = float(len(pred_types)) and float( len(gold_types.intersection(pred_types))) / len(pred_types)
    types_r = float(len(gold_types)) and float( len(gold_types.intersection(pred_types))) / len(gold_types)
    types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
    results['TOTAL']['TYPES']['prec'] = types_p
    results['TOTAL']['TYPES']['rec'] = types_r
    results['TOTAL']['TYPES']['f1'] = types_f

    print("STRICT:\n ",strict_p,strict_r,strict_f)
    print("SPANS:\n ",spans_p,spans_r,spans_f)
    print("TYPES:\n ",types_p,types_r,types_f)
    return(results)






def on_data_flausch_spans(gold, predicted):

    gold['cid']= gold['document']+"_"+gold['comment_id'].apply(str)
    predicted['cid']= predicted['document']+"_"+predicted['comment_id'].apply(str)


    # annotation sets (predicted)
    pred_spans = Multiset()
    pred_spans_loose = Multiset()
    pred_types = Multiset()

    # annotation sets (gold)
    gold_spans = Multiset()
    gold_spans_loose = Multiset()
    gold_types = Multiset()

    for row in predicted.itertuples(index=False):
        pred_spans.add((row.cid,row.type,row.start,row.end))
        pred_spans_loose.add((row.cid,row.start,row.end))
        pred_types.add((row.cid,row.type))
    for row in gold.itertuples(index=False):
        gold_spans.add((row.cid,row.type,row.start,row.end))
        gold_spans_loose.add((row.cid,row.start,row.end))
        gold_types.add((row.cid,row.type))

    # precision = true_pos / true_pos + false_pos
    # recall = true_pos / true_pos + false_neg
    # f_1 = 2 * prec * rec / (prec + rec)

    results = {'TOTAL': {'STRICT': {},'SPANS': {},'TYPES': {}}}

    # Overall evaluation
    # strict: spans + type must match
    strict_p = float(len(pred_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(pred_spans)
    strict_r = float(len(gold_spans)) and float( len(gold_spans.intersection(pred_spans))) / len(gold_spans)
    strict_f = (strict_p + strict_r) and 2 * strict_p * strict_r / (strict_p + strict_r)
    results['TOTAL']['STRICT']['prec'] = strict_p
    results['TOTAL']['STRICT']['rec'] = strict_r
    results['TOTAL']['STRICT']['f1'] = strict_f

    # spans: spans must match
    spans_p = float(len(pred_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(pred_spans_loose)
    spans_r = float(len(gold_spans_loose)) and float( len(gold_spans_loose.intersection(pred_spans_loose))) / len(gold_spans_loose)
    spans_f = (spans_p + spans_r) and 2 * spans_p * spans_r / (spans_p + spans_r)
    results['TOTAL']['SPANS']['prec'] = spans_p
    results['TOTAL']['SPANS']['rec'] = spans_r
    results['TOTAL']['SPANS']['f1'] = spans_f

    # detection mode: only types must match (per post)
    types_p = float(len(pred_types)) and float( len(gold_types.intersection(pred_types))) / len(pred_types)
    types_r = float(len(gold_types)) and float( len(gold_types.intersection(pred_types))) / len(gold_types)
    types_f = (types_p + types_r) and 2 * types_p * types_r / (types_p + types_r)
    results['TOTAL']['TYPES']['prec'] = types_p
    results['TOTAL']['TYPES']['rec'] = types_r
    results['TOTAL']['TYPES']['f1'] = types_f

    print("STRICT:\n ",strict_p,strict_r,strict_f)
    print("SPANS:\n ",spans_p,spans_r,spans_f)
    print("TYPES:\n ",types_p,types_r,types_f)
    return(results)




## Make submission file function

In [27]:
import csv
import zipfile
import os


def make_submission_task1(data, name):
    submission = data[["document","comment_id","flausch"]]
    for f in submission["flausch"]:
        if (f != "yes" and f != "no"):
            raise Exception("flausch prediction missing")
    csv_filename = submission_path + name + ".csv"
    zip_filename = submission_path + name + ".zip"
    submission.to_csv(csv_filename, index=False, quoting=csv.QUOTE_ALL)
    # make zip file
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
      zipf.write(csv_filename, os.path.basename(csv_filename))

    print(f"Submission saved to {csv_filename} and zipped to {zip_filename}")


In [28]:
import zipfile
import os

def make_submission_task2(data, name):
    submission = data[["document","comment_id","type","start","end"]]
    for f in submission["start"]:
        if not type(f) is int:
            raise Exception("start prediction missing")
    for f in submission["end"]:
        if not type(f) is int:
            raise Exception("end prediction missing")
    csv_filename = submission_path + name + ".csv"
    zip_filename = submission_path + name + ".zip"
    submission.to_csv(csv_filename, index=False, quoting=csv.QUOTE_ALL)
    # make zip file
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
      zipf.write(csv_filename, os.path.basename(csv_filename))

    print(f"Submission saved to {csv_filename} and zipped to {zip_filename}")





## Task 1

#### BERT model gBert-large: binary classifier: yes/no flausch

In [29]:
from datasets import Dataset

test_task1_dataset = Dataset.from_pandas(test_data_task1)
test_task1_dataset

Dataset({
    features: ['document', 'comment_id', 'comment', 'flausch', 'translated', 'spelling_corrected', 'sentiment_orig', 'sentiment_spelling_corrected', 'sentiment_translated', 'sentiment_anger', 'sentiment_disgust', 'sentiment_fear', 'sentiment_joy', 'sentiment_neutral', 'sentiment_sadness', 'sentiment_surprise', 'id'],
    num_rows: 3706
})

In [30]:
def apply_task1_pipeline(batch,pipeline,column,tokenizer_kwargs):
    texts = batch[column]
    results_for_batch = pipeline(texts, **tokenizer_kwargs)
    predictions = []
    top_scores = [] # List to store the scores of the top prediction

    for top_pred_item in results_for_batch:
        # top_pred_item ist ein Dictionary
        # Beispiel: {'label': 'POSITIVE', 'score': 0.99}
        predictions.append(top_pred_item['label'])
        top_scores.append(top_pred_item['score'])

    # Return a dictionary with the new columns, matching the batch size
    return {'prediction': predictions, 'score': top_scores}



In [31]:
from transformers import pipeline

def make_predictions_task1(model_name, column):
    pipe = pipeline("text-classification", model=model_name)
    tokenizer_kwargs = {'padding':True,'truncation':True,'max_length':512}
        # for our test set 12 minutes on local machine on colab ~ 1 minute
    prediction_dataset = test_task1_dataset.map(
        apply_task1_pipeline,
        batched=True,
        batch_size=16,
        num_proc=1,
        fn_kwargs={"pipeline": pipe, "column": column, "tokenizer_kwargs": tokenizer_kwargs}
    )
    return prediction_dataset

In [32]:
file_gold = path + "gold_data_task1.csv"

In [37]:
# prediction with BERT model
# Careful: takes some time to predict with all models on all columns

checkpoint = "Wiebke/results_flausch_classification_"

if not os.path.exists(path + "features.csv"):
    features = test_data_task1[["id","document","comment_id"]].copy().reset_index(drop=True)
else:
    features = pd.read_csv(path + "features.csv").reset_index(drop=True)


# german models on spelling corrected comments and on original comments
for my_model in ["bert-base-german-cased", "gbert-large"]:
    for column in ["spelling_corrected", "comment"]:
        # German models gBERT-large and bert-base-german-cased and multilingual model roberta-large
        model_name = checkpoint + my_model + "_" + column
        prediction_dataset  = make_predictions_task1(model_name, column)
        submission = pd.DataFrame(prediction_dataset)
        submission["flausch"] = submission["prediction"]
        make_submission_task1(submission,"test_submission_task1_" + my_model + "_" + column)
        file_pred = submission_path + "test_submission_task1_"  + my_model  + "_" + column + ".csv"
        print("\nEvaluation for configuration", my_model, column)
        binary_eval(file_gold, file_pred)
        # update features
        features[my_model + "_yes_" + column] =  float('nan')
        submission["score"] = submission["score"].astype(float)
        submission.loc[submission["prediction"] == "no", "score"] = 1-submission["score"]
        score_dict = dict(zip(submission["id"], submission["score"]))
        features[my_model + "_yes_" + column] = features["id"].map(score_dict)




# english model on translated comments
my_model =  "roberta-large"
column = "translated"
model_name = model= checkpoint + my_model + "_" + column
pipe = pipeline("text-classification", model=model_name)
tokenizer_kwargs = {'padding':True,'truncation':True,'max_length':512}
prediction_dataset  = make_predictions_task1(model_name, column)
submission = pd.DataFrame(prediction_dataset)
submission["flausch"] = submission["prediction"]
make_submission_task1(submission,"test_submission_task1_" + my_model + "_" + column)
file_pred = submission_path + "test_submission_task1_"  + my_model  + "_" + column + ".csv"
print("\nEvaluation for configuration", my_model, column)
binary_eval(file_gold, file_pred)
# update features
features[my_model + "_yes_" + column] =  float('nan')
submission["score"] = submission["score"].astype(float)
submission.loc[submission["prediction"] == "no", "score"] = 1-submission["score"]
score_dict = dict(zip(submission["id"], submission["score"]))
features[my_model + "_yes_" + column] = features["id"].map(score_dict)




Device set to use cuda:0


Map:   0%|          | 0/3706 [00:00<?, ? examples/s]

Submission saved to /content/drive/MyDrive/FlauschData/submission/test_submission_task1_bert-base-german-cased_spelling_corrected.csv and zipped to /content/drive/MyDrive/FlauschData/submission/test_submission_task1_bert-base-german-cased_spelling_corrected.zip

Evaluation for configuration bert-base-german-cased spelling_corrected
TASK 1: BINARY CLASSIFICATION 
Precision:	 0.8810 
Recall:		 0.8793 
F-score:	 0.8802


Device set to use cuda:0


Map:   0%|          | 0/3706 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/3706 [00:00<?, ? examples/s]

Submission saved to /content/drive/MyDrive/FlauschData/submission/test_submission_task1_gbert-large_comment.csv and zipped to /content/drive/MyDrive/FlauschData/submission/test_submission_task1_gbert-large_comment.zip

Evaluation for configuration gbert-large comment
TASK 1: BINARY CLASSIFICATION 
Precision:	 0.8805 
Recall:		 0.9320 
F-score:	 0.9055


Device set to use cuda:0
Device set to use cuda:0


Map:   0%|          | 0/3706 [00:00<?, ? examples/s]

Submission saved to /content/drive/MyDrive/FlauschData/submission/test_submission_task1_roberta-large_translated.csv and zipped to /content/drive/MyDrive/FlauschData/submission/test_submission_task1_roberta-large_translated.zip

Evaluation for configuration roberta-large translated
TASK 1: BINARY CLASSIFICATION 
Precision:	 0.8853 
Recall:		 0.8649 
F-score:	 0.8750


In [42]:

features = features.drop(columns=[x for x in features.columns if x.startswith("Unnamed")])

features.to_csv(path + "features.csv")



### Compare BERT classification models


In [45]:
features = pd.read_csv(path + "features.csv")

german_models = ["gbert-large", "bert-base-german-cased"]
english_models = ["roberta-large"]
german_columns = ["comment", "spelling_corrected"]
english_columns = ["translated"]
testdata = features[features["set"]=="test"].reset_index(drop=True)

gold = pd.read_csv(path + "gold_data_task1.csv")
gold = make_id_column(gold)
gold = gold[["id", "flausch"]]
testdata = testdata.merge(gold, on="id")

test = testdata.copy()

cnames = []
for m in german_models:
    for c in german_columns:
        cname = m + "_yes_" + c
        cnames.append(cname)

for m in english_models:
    for c in english_columns:
        cname = m + "_yes_" + c
        cnames.append(cname)

for c in cnames:
    test[c] = test[c].apply(lambda x: "yes" if x > 0.5 else "no")

test.head(3)


,Unnamed: 0,id,set,gbert-large_yes_comment,bert-base-german-cased_yes_comment,bert-base-german-cased_yes_spelling_corrected,gbert-large_yes_spelling_corrected,roberta-large_yes_translated,flausch
0,4,NDY-003_5,test,yes,yes,yes,yes,yes,yes
1,13,NDY-003_14,test,no,no,no,no,no,no
2,40,NDY-003_41,test,yes,yes,yes,yes,no,yes


In [46]:
import numpy as np
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, accuracy_score
for c in cnames:
    gold = test["flausch"]
    pred = test[c]
    print()
    print(c)
    print("weighted F1 score:", f1_score(gold,pred,average="weighted"))
    print("accuracy:", accuracy_score(gold,pred))
    print(classification_report(gold,pred))




gbert-large_yes_comment
weighted F1 score: 0.9456839415000668
accuracy: 0.9452239611440907
              precision    recall  f1-score   support

          no       0.97      0.95      0.96      2662
         yes       0.88      0.93      0.91      1044

    accuracy                           0.95      3706
   macro avg       0.93      0.94      0.93      3706
weighted avg       0.95      0.95      0.95      3706


gbert-large_yes_spelling_corrected
weighted F1 score: 0.9411003608844272
accuracy: 0.9409066378845116
              precision    recall  f1-score   support

          no       0.96      0.95      0.96      2662
         yes       0.89      0.91      0.90      1044

    accuracy                           0.94      3706
   macro avg       0.92      0.93      0.93      3706
weighted avg       0.94      0.94      0.94      3706


bert-base-german-cased_yes_comment
weighted F1 score: 0.9349608447726742
accuracy: 0.9349703184025904
              precision    recall  f1-score   su

In [47]:
id_false = test[test["gbert-large_yes_comment"] != test["flausch"]]["id"]
false = testdata[testdata["id"].isin(id_false)]
false

,Unnamed: 0,id,set,gbert-large_yes_comment,bert-base-german-cased_yes_comment,bert-base-german-cased_yes_spelling_corrected,gbert-large_yes_spelling_corrected,roberta-large_yes_translated,flausch
21,261,NDY-003_262,test,0.991680,0.989212,0.993828,0.991255,0.995281,no
22,263,NDY-003_264,test,0.990066,0.158032,0.317639,0.972652,0.994509,no
30,314,NDY-003_315,test,0.940336,0.487244,0.991803,0.989425,0.993041,no
87,943,NDY-003_944,test,0.970248,0.003784,0.004983,0.991856,0.992910,no
92,977,NDY-003_978,test,0.968321,0.001037,0.008553,0.532543,0.954395,no
...,...,...,...,...,...,...,...,...,...
3567,35681,NDY-252_132,test,0.716943,0.005428,0.113768,0.922663,0.022369,no
3615,36236,NDY-252_687,test,0.685844,0.932554,0.377382,0.032994,0.989608,no
3659,36639,NDY-274_91,test,0.943479,0.079387,0.151629,0.020367,0.990125,no
3679,36811,NDY-274_263,test,0.042193,0.000325,0.001814,0.013124,0.003413,yes


In [48]:
### train a metaclassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = testdata[cnames]
y = testdata["flausch"]

X_train, X_test, y_train, y_test =  train_test_split(X, y, test_size=0.2, random_state=42)



model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nRandom Forest")
print("F1 score weighted:",f1_score(y_test, y_pred,average="weighted"))
print("accuracy:",accuracy_score(y_test, y_pred))

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nDecision Tree")
print("F1 score weighted:",f1_score(y_test, y_pred,average="weighted"))
print("accuracy:",accuracy_score(y_test, y_pred))



model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nLogistic Regression")
print("F1 score weighted:",f1_score(y_test, y_pred,average="weighted"))
print("accuracy:",accuracy_score(y_test, y_pred))
print("coefficient    ", "configuration")
for i in range(len(X_test.columns)):
    print(model.coef_[0][i], " ", X_test.columns[i])


print(classification_report(y_test, y_pred))


Random Forest
F1 score weighted: 0.9538939248821557
accuracy: 0.954177897574124

Decision Tree
F1 score weighted: 0.9205443183562734
accuracy: 0.9204851752021563

Logistic Regression
F1 score weighted: 0.9593181690136667
accuracy: 0.9595687331536388
coefficient     configuration
2.6502891874395584   gbert-large_yes_comment
1.3588971413176498   gbert-large_yes_spelling_corrected
0.8798052532348425   bert-base-german-cased_yes_comment
1.003752660390435   bert-base-german-cased_yes_spelling_corrected
0.868199345440643   roberta-large_yes_translated
              precision    recall  f1-score   support

          no       0.97      0.98      0.97       536
         yes       0.94      0.91      0.93       206

    accuracy                           0.96       742
   macro avg       0.95      0.94      0.95       742
weighted avg       0.96      0.96      0.96       742



## Task 2

### Task2: Token classifier that uses BIO-scheme

BERT model gBERT-large: Token classification BIO scheme

In [ ]:
from transformers import pipeline
checkpoint = "Wiebke/flausch_span_gbert-large_all"

pipe = pipeline("token-classification", model=checkpoint)
pipe("das habt ihr großartig gmacht. Macht weiter so. Nächsten Monat wird es warm. Team Sommer!!!!")

In [ ]:
texts = ["Hunde sind Säugetiere", "großen Dank", "das habt ihr großartig gmacht. Macht weiter so. Nächsten Monat wird es warm. Team Sommer!!!!"]
pipe(texts)

In [ ]:
def apply_task2_pipeline(batch,pipeline,column):
    texts = batch[column]
    results_for_batch = pipeline(texts)

    types = []
    starts = []
    ends = []
    scores = []

    for single_text_result in results_for_batch:
        # single_text_result ist eine Liste von Dictionaries
        # Beispiel: [{'entity': 'B-gratitude', 'score': 0.87086165, 'index': 1,  'word': 'großen', 'start': 0,  'end': 6},
        #{'entity': 'I-gratitude','score': 0.78514546, 'index': 2,'word': 'Dank','start': 7,'end': 11}],
        type = []
        start = []
        end = []
        score = []
        for dic in single_text_result:
            type.append(dic['entity'])
            start.append(dic['start'])
            end.append(dic['end'])
            score.append(dic['score'])

        types.append(type)
        starts.append(start)
        ends.append(end)
        scores.append(score)

    # Return a dictionary with the new columns, matching the batch size
    return {'types': types, 'starts': starts, 'ends': ends, 'scores': scores}


In [ ]:
from datasets import Dataset

test_task2_dataset = Dataset.from_pandas(test_data_task2)


In [ ]:
test_task2_dataset

In [ ]:
prediction_dataset = test_task2_dataset.map(
    apply_task2_pipeline,
    batched=True,
    batch_size=16,
    num_proc=1,
    fn_kwargs={"pipeline": pipe, "column": "comment"}
)




In [ ]:
def map_to_type(string):
    return string.split("-")[1]



def merge_types(row):
    if len(row["types"]) == 0:
        return row
    current_t = map_to_type(row["types"][0])
    filter = [1]
    filter_end = [0]
    if len(row["types"]) > 1:
        for t in row["types"][1:]:
            if t == "I-" + current_t:
                filter.append(0)
                filter_end.append(0)
            else:
                filter_end[-1] = 2
                current_t = map_to_type(t)
                filter.append(1)
                filter_end.append(0)
        filter_end[-1] = 2
    filter_end[-1] = 2
    row["types"] = [row["types"][i] for i in range(len(filter)) if filter[i] == 1]
    row["starts"] = [row["starts"][i] for i in range(len(filter)) if filter[i] == 1]
    row["ends"] = [row["ends"][i] for i in range(len(filter_end)) if filter_end[i] == 2]
    if len(row["types"]) != len(row["starts"]):
        print("problem types-Länge ungleich starts-Länge")
    if len(row["types"]) != len(row["ends"]):
        print("problem types-Länge ungleich ends-Länge")
        print(filter)
        print(filter_end)
        print()
    return row



In [ ]:

def output_task2(df):
#    df["types"] = df["types"].apply(lambda x: list(map(map_to_type, x)))
    df = df.apply(merge_types, axis=1)
    output = []
    for i in range(df.shape[0]):
        types, starts, ends = df.loc[i][["types","starts","ends"]]
        if len(types) > 0:
            document, comment_id = df.loc[i][["document", "comment_id"]]
            for j in range(len(types)):
                output.append({"document": document,"comment_id": comment_id,
                               "type": map_to_type(types[j]), "start": starts[j],"end":ends[j]})
    return pd.DataFrame(output)



In [ ]:
import os

df = pd.DataFrame(prediction_dataset)
out_task2 = output_task2(df)
make_submission_task2(out_task2, "test_submission_task2_gbert")

In [ ]:
df.head(3)

In [ ]:
file_gold = path + "gold_data_task2.csv"
file_pred = submission_path + "test_submission_task2_gbert" + ".csv"
results = fine_grained_flausch_by_label(file_gold, file_pred)

typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()



In [ ]:
gold_data_task2.shape, out_task2.shape

#### Fehleranalyse (not ready)

In [ ]:
dfa= out_task2
dfb =  gold_data_task2

# Finde die Zeilen in df2, die auch in df1 sind
# Ein 'merge' mit 'indicator=True' zeigt an, woher die Zeilen kommen
merged_dfb = dfb.merge(dfa, how='left', indicator=True)
merged_dfa = dfa.merge(dfb, how='left', indicator=True)


# Filtere nur die Zeilen, die NICHT in df1 vorkommen
# '_merge' == 'left_only' bedeutet, die Zeile war nur in df2
dfb_cleaned_all_cols = merged_dfb[merged_dfb['_merge'] == 'left_only'].drop(columns=['_merge'])
dfa_cleaned_all_cols = merged_dfa[merged_dfa['_merge'] == 'left_only'].drop(columns=['_merge'])




In [ ]:
dfa.shape, dfa_cleaned_all_cols.shape, dfb.shape, dfb_cleaned_all_cols.shape

In [ ]:
dfa_cleaned_all_cols.loc[10:25]

In [ ]:
dfb_cleaned_all_cols.loc[10:25]

In [ ]:
document = "NDY-003"
comment_id = 76

# get prediction_dataset and gold_data_task2 mit document and comment_id

filtered_rows_gold = gold_data_task2[(gold_data_task2['document'] == document) & (gold_data_task2['comment_id'] == comment_id)]
print(filtered_rows_gold)

filtered_rows_out = out_task2[(out_task2['document'] == document) & (out_task2['comment_id'] == comment_id)]

print()
print(filtered_rows_out)

Beobachtung: eigene sind häufiger etwas zu lang.

### Task2: Alternative approach -- first flausch span detection then flausch span classification

In [ ]:
detector = "Wiebke/flausch_span_gbert-large_non_labeled_spans"
classifier = "Wiebke/results_flausch_classification_gbert-large_span_classifier"

# test whether a classifier that was trained on all spans (including not flausch spans performs better)
# result: no
#classifier = "Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan"

In [ ]:
from transformers import pipeline

pipe = pipeline("token-classification", model=detector)


In [ ]:
pipe("das habt ihr großartig gmacht. Macht weiter so. Nächsten Monat wird es warm. Team Sommer!!!!")

In [ ]:
test_task2_dataset = Dataset.from_pandas(test_data_task2)

In [ ]:
prediction_dataset = test_task2_dataset.map(
    apply_task2_pipeline,
    batched=True,
    batch_size=16,
    num_proc=1,
    fn_kwargs={"pipeline": pipe, "column": "comment"}
)


In [ ]:
df = pd.DataFrame(prediction_dataset)
out_task2 = output_task2(df)

In [ ]:
out_task2 = out_task2.rename(columns={"type": "temp_type"})

In [ ]:
test_data_task1.columns

In [ ]:
my_columns = ['document', 'comment_id', 'comment',  'id']
out_task2 = pd.merge(out_task2, test_data_task1[my_columns], on=["comment_id","document"], how = "left")

In [ ]:
out_task2["span"] = out_task2.apply(lambda row: row["comment"][row["start"]:(row["end"]+1)], axis=1)

In [ ]:
out_task2.head(3)

In [ ]:
pipe = pipeline("text-classification", model=classifier)


In [ ]:
# apply pipe to column "span" and write prediction into column "type"

def apply_span_classification_pipeline(row, pipeline):
    text = row['span']
    result = pipeline(text)
    # Assuming the pipeline returns a list of dictionaries and we need the label of the first one
    if result:
        return result[0]['label']
    return None

out_task2['type'] = out_task2.apply(lambda row: apply_span_classification_pipeline(row, pipe), axis=1)

In [ ]:
out_task2.head(3)

In [ ]:
make_submission_task2(out_task2, mode + "_test_submission_task2_gbert_2steps")

In [ ]:
# final submission for task 2
if mode == "test":
  make_submission_task2(out_task2, "task2-predicted")

In [ ]:
gold_data_task2 = pd.read_csv(path + "gold_data_task2.csv")
file_gold = path + "gold_data_task2.csv"
file_pred = submission_path + mode + "_test_submission_task2_gbert_2steps" + ".csv"
results = fine_grained_flausch_by_label(file_gold, file_pred)

typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()

In [ ]:
out_task2["type"].value_counts()

# Data statistics

In [ ]:
import pandas as pd
google_path = "Input/Data/"

our_path  = google_path + "train" +"/"
own_train_data_task1 = pd.read_json(our_path + "train_task1.json")
own_train_data_task2 = pd.read_json(our_path + "train_task2.json")
own_test_data_task1 = pd.read_json(our_path + "test_task1.json")
own_test_data_task2 = pd.read_json(our_path + "test_task2.json")



In [ ]:
test_path = google_path + "test_gold" +"/"
comp_test_data_task1 = pd.read_csv(test_path + "task1.csv")
comp_test_data_task2 = pd.read_csv(test_path + "task2.csv")
comp_test_comment = pd.read_csv(test_path + "comments.csv")
comp_train_task1 = pd.concat([own_train_data_task1,own_test_data_task1], ignore_index=True)
comp_train_task2 = pd.concat([own_train_data_task2,own_test_data_task2], ignore_index=True)

In [ ]:
print("own_train_data_task1",own_train_data_task1.shape)
print("own_test_data_task1",own_test_data_task1.shape)
print("own_train_data_task2",own_train_data_task2.shape)
print("own_test_data_task2",own_test_data_task2.shape)
print("comp_test_data_task1", comp_test_data_task1.shape)
print("comp_test_data_task2", comp_test_data_task2.shape)
print("comp_train_task1", comp_train_task1.shape   )


In [ ]:
print("task1")
print("ratio flausch in own train data", own_train_data_task1[own_train_data_task1["flausch"]=="yes"].shape[0]/own_train_data_task1.shape[0])
print("ratio flausch in own test data", own_test_data_task1[own_test_data_task1["flausch"]=="yes"].shape[0]/own_test_data_task1.shape[0])
print("ratio flausch in gold competition train data", comp_train_task1[comp_train_task1["flausch"]=="yes"].shape[0]/comp_train_task1.shape[0])
print("ratio flausch in gold competition test data", comp_test_data_task1[comp_test_data_task1["flausch"]=="yes"].shape[0]/comp_test_data_task1.shape[0])

print("")
print("average length of comments in own train data", own_train_data_task1["comment"].apply(lambda x: len(x)).mean())
print("average length of comments in own test data", own_test_data_task1["comment"].apply(lambda x: len(x)).mean())
print("average length of comments in gold competition train data", comp_train_task1["comment"].apply(lambda x: len(x)).mean())
print("average length of comments in gold competition test data", comp_test_comment["comment"].apply(lambda x: len(x)).mean())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import numpy as np
own_train_lengths = own_train_data_task1["comment"].apply(len)
own_test_lengths = own_test_data_task1["comment"].apply(len)
comp_test_lengths = comp_test_comment["comment"].apply(len)

for lengths in [own_train_lengths, own_test_lengths, comp_test_lengths]:
    print("Mean length:", lengths.mean())
    print("Median length:", lengths.median())
    print("Standard deviation:", lengths.std())
    print("Minimum length:", lengths.min())
    print("Maximum length:", lengths.max())
    print()


# DataFrame für seaborn
data = pd.DataFrame({
    "Length": np.concatenate([own_train_lengths, own_test_lengths, comp_test_lengths]),
    "Dataset": (["Own Train"] * len(own_train_lengths)) +
               (["Own Test"] * len(own_test_lengths)) +
               (["Competition Test"] * len(comp_test_lengths))
})

# Plot
plt.figure(figsize=(8, 5))
sns.boxplot(x="Dataset", y="Length", data=data, palette="pastel", showfliers=False)

plt.ylabel("Comment Length (in tokens)")
plt.title("Distribution of Comment Lengths Across Datasets (Task 1)")
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
type_list = own_train_data_task2["type"].unique()

In [ ]:
print("task2")
type_distribution = own_train_data_task2["type"].value_counts()/own_train_data_task2.shape[0]
print("distribution in own train data")
print(type_distribution.reindex(type_list))
type_distribution = own_test_data_task2["type"].value_counts()/own_test_data_task2.shape[0]
print("distribution in own test data")
print(type_distribution.reindex(type_list))
type_distribution = comp_test_data_task2["type"].value_counts()/comp_test_data_task2.shape[0]
print("distribution in gold competition test data")
print(type_distribution.reindex(type_list))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === Assuming your DataFrames are already loaded ===
# - own_train_data_task2, own_test_data_task2, comp_test_data_task2
# - own_train_data_task1, own_test_data_task1, comp_test_data_task1

# Map of data sources and labels
datasets = {
    "Own Train": {
        "spans_df": own_train_data_task2,
        "comments_df": own_train_data_task1
    },
    "Own Test": {
        "spans_df": own_test_data_task2,
        "comments_df": own_test_data_task1
    },
    "Competition Test": {
        "spans_df": comp_test_data_task2,
        "comments_df": comp_test_data_task1
    }
}

# Step 1: Compute normalized counts of span types scaled by spans per comment
span_type_ratios = {}

for label, data in datasets.items():
    spans_df = data["spans_df"]
    comments_df = data["comments_df"]

    num_spans = len(spans_df)
    num_comments = len(comments_df)

    spans_per_comment = num_spans / num_comments

    # Normalize type distribution
    type_counts = spans_df["type"].value_counts(normalize=True).to_dict()

    # Multiply each type by spans per comment
    scaled_distribution = {t: ratio * spans_per_comment for t, ratio in type_counts.items()}

    span_type_ratios[label] = scaled_distribution

# Step 2: Create DataFrame for plotting
df = pd.DataFrame(span_type_ratios).fillna(0).T

# Optional: consistent order of columns (alphabetical or custom)
df = df[sorted(df.columns)]

# Step 3: Plot
ax = df.plot(kind="bar", stacked=True, figsize=(6, 6), colormap="Paired", edgecolor="black") #Paired


# Get legend handles and labels
handles, labels = ax.get_legend_handles_labels()

# Reverse the order
ax.legend(reversed(handles), reversed(labels), title="Span Types")

#plt.title("Span Type Distribution (Normalized by Spans per Comment)", fontsize=11)
plt.ylabel("Average Spans per Comment")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.legend(reversed(handles), reversed(labels),  bbox_to_anchor=(0, 1), loc='upper left',fontsize=9)
plt.tight_layout()
plt.savefig("span_type_distribution_normalized_by_spans_per_comment.pdf", bbox_inches='tight', dpi=300)
plt.show()



In [ ]:
print("spans per comment in own train data", own_train_data_task2.shape[0]/own_train_data_task1.shape[0])
print("spans per comment in own test data", own_test_data_task2.shape[0]/own_test_data_task1.shape[0])
print("spans per comment in gold competition train data", comp_train_task2.shape[0]/comp_train_task1.shape[0])
print("spans per comment in gold competition test data", comp_test_data_task2.shape[0]/comp_test_data_task1.shape[0])

In [ ]:
own_train_data_task2.head(3)

In [ ]:
print("Average length of spans in own train data", (own_train_data_task2["end"] - own_train_data_task2["start"]).mean())
print("Average length of spans in own test data", (own_test_data_task2["end"] - own_test_data_task2["start"]).mean())
print("Average length of spans in gold competition test data", (comp_test_data_task2["end"] - comp_test_data_task2["start"]).mean())



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import numpy as np
own_train_lengths = own_train_data_task2["end"] - own_train_data_task2["start"]
own_test_lengths = own_test_data_task2["end"] - own_test_data_task2["start"]
comp_test_lengths = comp_test_data_task2["end"] - comp_test_data_task2["start"]

for lengths in [own_train_lengths, own_test_lengths, comp_test_lengths]:
    print("Mean length:", lengths.mean())
    print("Median length:", lengths.median())
    print("Standard deviation:", lengths.std())
    print("Minimum length:", lengths.min())
    print("Maximum length:", lengths.max())
    print()


# DataFrame für seaborn
data = pd.DataFrame({
    "Length": np.concatenate([own_train_lengths, own_test_lengths, comp_test_lengths]),
    "Dataset": (["Own Train"] * len(own_train_lengths)) +
               (["Own Test"] * len(own_test_lengths)) +
               (["Competition Test"] * len(comp_test_lengths))
})

# Plot
plt.figure(figsize=(8, 5))
sns.boxplot(x="Dataset", y="Length", data=data, palette="pastel", showfliers=False)

plt.ylabel("Comment Length (in tokens)")
plt.title("Distribution of Comment Lengths Across Datasets (Task 1)")
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


# Evaluate Fine-Tuned models for subtask 2

## Wiebke/flausch_span_gbert-large_all



In [ ]:
checkpoint = "Wiebke/flausch_span_gbert-large_all"
pipe = pipeline("token-classification", model=checkpoint)


### own test

In [ ]:
own_test_comment_dataset = Dataset.from_pandas(own_test_data_task1)

In [ ]:
prediction_dataset = own_test_comment_dataset.map(
    apply_task2_pipeline,
    batched=True,
    batch_size=16,
    num_proc=1,
    fn_kwargs={"pipeline": pipe, "column": "comment"}
)


In [ ]:
df = pd.DataFrame(prediction_dataset)
out_task2 = output_task2(df)
gold_data_task2 = own_test_data_task2[["document", "comment_id", "type", "start","end"]]


In [ ]:
results = on_data_fine_grained_flausch_by_label(gold_data_task2, out_task2)

In [ ]:

typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()


### competition test

In [ ]:
comp_test_comment_dataset = Dataset.from_pandas(comp_test_comment)

In [ ]:
prediction_dataset = comp_test_comment_dataset.map(
    apply_task2_pipeline,
    batched=True,
    batch_size=16,
    num_proc=1,
    fn_kwargs={"pipeline": pipe, "column": "comment"}
)


In [ ]:
df = pd.DataFrame(prediction_dataset)
out_task2 = output_task2(df)
gold_data_task2 = comp_test_data_task2[["document", "comment_id", "type", "start","end"]]


In [ ]:
results = on_data_fine_grained_flausch_by_label(gold_data_task2, out_task2)

In [ ]:

typeList = gold_data_task2["type"].unique()
all = []
for t in typeList:
    print(t)
    print(pd.DataFrame(results[t]))
    print()


## Wiebke/flausch_span_gbert-large_non_labeled_spans

In [ ]:
checkpoint = "Wiebke/flausch_span_gbert-large_non_labeled_spans"
pipe = pipeline("token-classification", model=checkpoint)


### own test

In [ ]:
own_test_comment_dataset = Dataset.from_pandas(own_test_data_task1)

In [ ]:
prediction_dataset = own_test_comment_dataset.map(
    apply_task2_pipeline,
    batched=True,
    batch_size=16,
    num_proc=1,
    fn_kwargs={"pipeline": pipe, "column": "comment"}
)


In [ ]:
df = pd.DataFrame(prediction_dataset)
out_task2 = output_task2(df)
gold_data_task2 = own_test_data_task2[["document", "comment_id", "type", "start","end"]]
gold_data_task2["type"] = ["span" for i in range(gold_data_task2.shape[0])]

In [ ]:
results = on_data_flausch_spans(gold_data_task2, out_task2)

### competition test

In [ ]:
comp_test_comment_dataset = Dataset.from_pandas(comp_test_comment)

In [ ]:
prediction_dataset = comp_test_comment_dataset.map(
    apply_task2_pipeline,
    batched=True,
    batch_size=16,
    num_proc=1,
    fn_kwargs={"pipeline": pipe, "column": "comment"}
)


In [ ]:
df = pd.DataFrame(prediction_dataset)
out_task2 = output_task2(df)
out_task2["type"] = ["span" for i in range(out_task2.shape[0])]
gold_data_task2 = comp_test_data_task2[["document", "comment_id", "type", "start","end"]]
gold_data_task2["type"] = ["span" for i in range(gold_data_task2.shape[0])]

In [ ]:
results = on_data_flausch_spans(gold_data_task2, out_task2)

## Wiebke/results_flausch_classification_gbert-large_span_classifier

In [ ]:
checkpoint = "Wiebke/results_flausch_classification_gbert-large_span_classifier"
pipe = pipeline("text-classification", model=checkpoint)

### own test

In [ ]:
out_task2 = own_test_data_task2[["span","type"]].copy()
out_task2['type'] = out_task2.apply(lambda row: apply_span_classification_pipeline(row, pipe), axis=1)

In [ ]:
gold_data_task2 = own_test_data_task2[["span","type"]].copy()

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(gold_data_task2["type"], out_task2["type"]))

### competition test

In [ ]:
#
comp_test_data_task2 = comp_test_data_task2.merge(comp_test_comment, on= ["document", "comment_id"]).copy()
comp_test_data_task2["span"] = comp_test_data_task2.apply(lambda row: row["comment"][row["start"]:(row["end"]+1)], axis=1)

In [ ]:
comp_test_task2_dataset = Dataset.from_pandas(comp_test_data_task2[["span","type"]])


In [ ]:
out_task2 = comp_test_data_task2[["span","type"]].copy()
out_task2['type'] = out_task2.apply(lambda row: apply_span_classification_pipeline(row, pipe), axis=1)

In [ ]:
gold_data_task2 = comp_test_data_task2[["span","type"]].copy()

In [ ]:
print(classification_report(gold_data_task2["type"], out_task2["type"]))

## Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan

In [ ]:
checkpoint = "Wiebke/task2_flausch_classification_gbert-large_span_classifier_with_nonspan"
pipe = pipeline("text-classification", model=checkpoint)

### own test

In [ ]:
own_test_data_task2_none = pd.read_csv(path + "phrases_with_labels_inlcuding_not_flausch_test.csv")
own_test_data_task2_none["label"].value_counts()

In [ ]:
out_task2 = own_test_data_task2_none.copy()
out_task2 = out_task2.rename(columns={"phrase": "span"})


In [ ]:
out_task2["span"] = out_task2["span"].apply(lambda x: str("") if x == "nan" else str(x))


In [ ]:
out_task2['type'] = out_task2.apply(lambda row: apply_span_classification_pipeline(row, pipe), axis=1)

In [ ]:
gold_data_task2 = out_task2[["span","label"]].copy()
gold_data_task2 = gold_data_task2.rename(columns={"label": "type"})

In [ ]:
print(classification_report(gold_data_task2["type"], out_task2["type"]))

# miscellaneous

In [ ]:
import pandas as pd
google_path = "Input/Data/"

our_path  = google_path + "train" +"/"
own_train_spans = pd.read_csv(our_path + "phrases_with_labels_inlcuding_not_flausch_train.csv")
own_test_spans = pd.read_csv(our_path + "phrases_with_labels_inlcuding_not_flausch_test.csv")



In [ ]:
own_train_spans["label"].value_counts(), own_test_spans["label"].value_counts()